# Teks — Praproses lengkap: periksa → bersihkan → ukur → simpan `train_bersih.csv`
**Kapan dipakai:** paling awal, sebelum `baseline_oof` / `finetune_oof` / `stacking_teks_tabular`.
Untuk data teks apa pun, dengan atau tanpa kolom tabel di sampingnya.

**Alur (Run All, lalu baca tabelnya dari atas ke bawah):**

| bagian | isi | yang kamu putuskan |
|---|---|---|
| 1 | periksa teks: 20 jenis kotoran, train % vs test % | kotoran mana yang nyata ada |
| 2 | awalan/akhiran berulang (nama sumber, hak cipta) vs label | `STRIP_PATTERNS` |
| 3 | bersih RINGAN (mojibake, HTML, unicode, pola berulang, spasi): aman untuk semua model | – |
| 4 | bersih BERAT (huruf kecil, URL, emoji, slang, stopword, stemming): hanya untuk TF-IDF | – |
| 5 | **ukur tiap langkah** dengan model dan fold yang sama | matikan langkah yang tidak naik |
| 6 | kolom tabel: kosong tersamar, angka/tanggal tertulis sebagai teks, kategori | daftar kolom |
| 7 | baris: teks kosong, kembar, kembar beda label | `DROP_...` |
| 8 | simpan `train_bersih.csv` + `test_bersih.csv` + `langkah.txt` | – |

**Hasil:** kolom teks asli diisi versi RINGAN (untuk BERT), kolom baru `<kolom>_tfidf` berisi
versi BERAT (untuk TF-IDF), kolom `meta_*` (panjang, kapital, tanda baca) untuk model
tabel. Nama kolom asli tidak berubah, baris test tidak pernah dibuang.

**Aman dari bocor:** semua langkah bekerja per baris (tidak belajar dari data), jadi boleh
dijalankan sebelum split. Yang belajar dari data (TF-IDF) ada di dalam fold di sel ukur.

**Saat lomba:** ubah sel Konfigurasi, hapus sel `HAPUS saat lomba`, Run All.
**Data demo:** SmSA (sentimen, 3 kelas), sengaja dikotori supaya tiap langkah terlihat kerjanya.

In [ ]:
import html
import os
import re
import time
import unicodedata
import warnings

import emoji
import numpy as np
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/praproses_lengkap"
DICTIONARY_FOLDER = "../../../data_latihan/preprocessing"      # kamus alay (ikut repo data)

ID_COL = "id"                    # GANTI saat lomba
TEXT_COLS = ["text"]             # GANTI saat lomba: boleh lebih dari satu kolom teks
TARGET = "label"                 # GANTI saat lomba
SCORING = "f1_macro"             # GANTI: metrik sel ukur ("accuracy", "f1_weighted", ...)
MEASURE_ROWS = 5000              # baris train untuk sel ukur (lebih banyak = lebih lama)
SEED = 42

## Baca data
File bukan CSV koma: pakai salah satu baris alternatif.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
# train = pd.read_csv(TRAIN_CSV, sep=";")                       # pemisah titik koma
# train = pd.read_csv(TRAIN_CSV, sep="\t")                      # .tsv
# train = pd.read_csv(TRAIN_CSV, encoding="latin-1")            # UnicodeDecodeError saat baca
# train = pd.read_excel(TRAIN_CSV)                              # .xlsx
# train = pd.read_json(TRAIN_CSV, lines=True)                   # .jsonl

only_train = []
for column in train.columns:
    if column not in test.columns:
        only_train.append(column)
only_test = []
for column in test.columns:
    if column not in train.columns:
        only_test.append(column)
print("train:", train.shape, "| test:", test.shape)
print("kolom hanya di train (target? jangan jadi fitur):", only_train)
print("kolom hanya di test:", only_test)
train.head(3)

## Kotori data demo (HAPUS saat lomba)
SmSA sudah bersih. Sel ini menambah kotoran yang pernah ditemui di data nyata: awalan sumber,
akhiran hak cipta, mojibake, HTML, URL, huruf berulang, emoji, karakter tak terlihat, dan tiga
kolom tabel kotor.

In [ ]:
def add_demo_dirt(table):
    row = np.arange(len(table))
    text = table[TEXT_COLS[0]].astype(str)
    text = text.where(row % 5 != 0, "Jakarta (SUMBER) - " + text)
    text = text.where(row % 4 != 0, text + " “kata sumber” COPYRIGHT © SUMBER 2021")
    text = text.where(row % 7 != 0, text + " <br>baca &amp\x3b bagikan https://contoh.id/a @admin")
    text = text.where(row % 6 != 0, text + " mantaaappp bgt 😡\u200b")
    broken = text.str.encode("utf-8").str.decode("mac_roman")
    table[TEXT_COLS[0]] = text.where(row % 8 != 0, broken)
    table["harga"] = np.array(["Rp 1.250.000", "750rb", "-", "2,5 jt"])[row % 4]
    table["tanggal"] = np.array(["12/03/2024", "15/03/2024", "?"])[row % 3]
    table["kota"] = np.array([" Jakarta", "jakarta", "BANDUNG", "Bandung "])[row % 4]
    return table


train = add_demo_dirt(train)
test = add_demo_dirt(test)
train.head(3)

## 1. Periksa teks
`raw_train` / `raw_test` = semua kolom teks digabung (hanya untuk diperiksa dan diukur).

**Mojibake** = huruf rusak karena salah baca encoding (`“` terbaca `‚Äú` atau `â€œ`).
`fix_mojibake` membalik tiap rangkaian huruf non-ASCII, dan hanya memakainya bila
hasilnya UTF-8 yang sah, jadi teks yang sudah benar tidak berubah.
Wajib dijalankan SEBELUM huruf kecil (`lower()` mengubah
huruf rusaknya sehingga tidak bisa dibalik lagi).

In [ ]:
def join_text(table):
    joined = table[TEXT_COLS[0]].fillna("").astype(str)
    for column in TEXT_COLS[1:]:
        joined = joined + " " + table[column].fillna("").astype(str)
    return joined


def fix_run(match):
    run = match.group()
    for codec in ["cp1252", "mac_roman"]:
        raw = run.encode(codec, "ignore")
        fixed = raw.decode("utf-8", "ignore")
        if fixed and len(raw) == len(run) and fixed.encode("utf-8") == raw:
            return fixed
    return run


def fix_mojibake(text):
    # shortcut: hanya salah baca cp1252 / MacRoman; encoding lain tambahkan ke daftar codec
    return re.sub(r"[^\x00-\x7F]{2,}", fix_run, text)


raw_train = join_text(train)
raw_test = join_text(test)
print(fix_mojibake("kata ‚Äúsumber‚Äù dan â€œkutipanâ€œ tetap, café tetap café"))

In [ ]:
PATTERNS = {
    "URL": r"https?://\S+|www\.\S+",
    "mention @": r"@\w+",
    "hashtag #": r"#\w+",
    "tag HTML <..>": r"<[a-zA-Z/!][^<>]*>",
    "entitas HTML &..": r"&#?\w+\x3b",
    "huruf rusak \ufffd": "\ufffd",
    "karakter tak terlihat": r"[\u200b-\u200f\u2060\ufeff\u00a0\u00ad\x00-\x08\x0b\x0c\x0e-\x1f]",
    "\\n tertulis sebagai teks": r"\\[nrt]",
    "baris baru / tab": r"[\n\r\t]",
    "huruf berulang 3+": r"([a-zA-Z])\1{2,}",
    "emoji / simbol": r"[\U0001F000-\U0001FAFF☀-➿]",
    "angka": r"\d",
    "list sebagai teks ['a', ..]": r"^\s*\[.*\]\s*$",
    "aksara bukan Latin": r"[Ѐ-ӿ؀-ۿ぀-ヿ一-鿿가-힯]",
    "kata Indonesia (yang/dan/di)": r"(?i)\b(?:yang|dan|di|tidak)\b",
    "kata Inggris (the/and/of)": r"(?i)\b(?:the|and|of|is)\b",
}


def share(mask):
    return round(100 * mask.mean(), 1)


print("pola diperiksa:", len(PATTERNS))

In [ ]:
def masks_of(texts):
    masks = {}
    for name, pattern in PATTERNS.items():
        masks[name] = texts.str.contains(pattern, regex=True)
    masks["mojibake (bisa diperbaiki)"] = texts.apply(fix_mojibake) != texts
    masks["HURUF BESAR semua"] = texts.str.isupper()
    masks["kosong"] = texts.str.strip() == ""
    masks["<= 2 kata"] = texts.str.split().str.len() <= 2
    return masks


def audit_text(train_text, test_text):
    test_masks = masks_of(test_text)
    rows = []
    for name, in_train in masks_of(train_text).items():
        example = ""
        if in_train.any():
            example = train_text[in_train].iloc[0][:80]
        rows.append({"cek": name, "train %": share(in_train),
                     "test %": share(test_masks[name]), "contoh": example})
    return pd.DataFrame(rows)


# Baca: baris dengan % > 0 = kotoran yang nyata ada. train % jauh beda dari test % = test berbeda
# gaya dari train (hati-hati, skor validasi bisa menipu).
audit_before = audit_text(raw_train, raw_test)
audit_before

In [ ]:
length_rows = []
for name, texts in [("train", raw_train), ("test", raw_test)]:
    words = texts.str.split().str.len()
    length_rows.append({"bagian": name, "median kata": words.median(), "p95": words.quantile(0.95),
                        "maks": words.max(), "> 400 kata %": share(words > 400)})
# > 400 kata: BERT (maks 512 token) memotong teks ini -> lihat contoh/text/dokumen_panjang
pd.DataFrame(length_rows)

## 2. Awalan dan akhiran berulang
Potongan yang sama di awal/akhir banyak teks (nama kantor berita, `COPYRIGHT ...`, tanda tangan,
`Baca juga`) bukan isi. Tabel di bawah: potongan tersering, porsinya di train dan test, dan label
yang paling sering menyertainya.

Cara membaca `porsi label %`:
- mendekati porsi kelas terbesar → potongan netral, buang (tulis polanya di `STRIP_PATTERNS`);
- jauh lebih tinggi (mis. 95%) → potongan itu membocorkan label (sumber data per kelas berbeda).
  Kalau `test %` kira-kira sama dengan `train %`, bocoran itu juga ada di test: boleh dibiarkan
  dan catat untuk slide. Kalau di test tidak ada: WAJIB dibuang, kalau tidak skor CV bohong.

In [ ]:
EDGE_WORDS = 4   # GANTI: berapa kata di ujung teks yang dilihat


def edge_pieces(texts, from_end, per_word):
    edges = texts.str.split().str[:EDGE_WORDS]
    if from_end:
        edges = texts.str.split().str[-EDGE_WORDS:]
    if per_word:
        edges = edges.explode()          # tiap kata dihitung sendiri (awalan dengan kota berganti)
    else:
        edges = edges.str.join(" ")
    pieces = pd.DataFrame({"row": edges.index, "piece": edges.to_numpy()})
    return pieces.drop_duplicates()


def edge_table(train_pieces, test_pieces):
    rows = []
    counts = train_pieces["piece"].value_counts()
    for piece, count in counts.head(6).items():
        with_piece = train_pieces.loc[train_pieces["piece"] == piece, "row"]
        labels = train.loc[with_piece, TARGET].value_counts(normalize=True)
        in_test = (test_pieces["piece"] == piece).sum()
        rows.append({"potongan": piece, "train %": round(100 * count / len(train), 1),
                     "test %": round(100 * in_test / len(test), 1),
                     "label terbanyak": labels.index[0],
                     "porsi label %": round(100 * labels.iloc[0], 1)})
    return pd.DataFrame(rows)


print("porsi kelas terbesar %:", round(100 * train[TARGET].value_counts(normalize=True).iloc[0], 1))

In [ ]:
VIEWS = [("AWALAN, frasa", False, False), ("AWALAN, per kata", False, True),
         ("AKHIRAN, frasa", True, False), ("AKHIRAN, per kata", True, True)]
for title, from_end, per_word in VIEWS:
    print(title)
    table = edge_table(edge_pieces(raw_train, from_end, per_word),
                       edge_pieces(raw_test, from_end, per_word))
    print(table.to_string(index=False), "\n")

## 3. Bersih RINGAN (untuk semua model, termasuk BERT)
Hanya memperbaiki yang rusak dan membuang yang bukan isi. Huruf besar, tanda baca, dan emoji
tetap, karena BERT memakainya. Matikan satu langkah = beri `#` di barisnya pada `LIGHT_STEPS`.

In [ ]:
STRIP_PATTERNS = [                       # GANTI: regex dari tabel awalan/akhiran, [] = tidak ada
    r"^\s*\w+ \(SUMBER\) - ",            # awalan "Kota (SUMBER) - "
    r"COPYRIGHT © SUMBER \d{4}",         # akhiran hak cipta
]
QUOTES = str.maketrans({"“": '"', "”": '"', "‘": "'", "’": "'", "–": "-", "—": "-"})
INVISIBLE = r"[\u200b-\u200f\u2060\ufeff\u00ad\x00-\x08\x0b\x0c\x0e-\x1f]"


def strip_html(text):
    text = html.unescape(text)                              # entitas HTML -> huruf
    return re.sub(r"<[a-zA-Z/!][^<>]*>", " ", text)         # <br>, <p class=..>, </a>


def normalize_unicode(text):
    text = unicodedata.normalize("NFKC", text)              # huruf lebar/gaya -> huruf biasa
    text = re.sub(INVISIBLE, "", text)                      # zero-width, BOM, karakter kontrol
    text = re.sub(r"\\[nrt]", " ", text)                    # "\n" yang tertulis sebagai dua huruf
    return text.translate(QUOTES)                           # kutip miring -> kutip biasa


def strip_patterns(text):
    for pattern in STRIP_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.I)
    return text


print(strip_patterns(normalize_unicode(strip_html("Solo (SUMBER) - a&amp\x3bb<br>"))))

In [ ]:
def squeeze_space(text):
    return " ".join(text.split())


LIGHT_STEPS = [
    ("mojibake", fix_mojibake),
    ("html", strip_html),
    ("unicode", normalize_unicode),
    ("pola berulang", strip_patterns),
    ("spasi", squeeze_space),
]
step_log = []


def apply_steps(texts, steps, part):
    stages = []
    for name, function in steps:
        cleaned = texts.apply(function)
        step_log.append({"langkah": name, "bagian": part,
                         "baris berubah": int((cleaned != texts).sum())})
        texts = cleaned
        stages.append((name, texts))
    return texts, stages


light_train, light_stages = apply_steps(raw_train, LIGHT_STEPS, "train")
light_test, _ = apply_steps(raw_test, LIGHT_STEPS, "test")
for position in np.where(light_train != raw_train)[0][:3]:
    print("SEBELUM:", raw_train.iloc[position][:150])
    print("SESUDAH:", light_train.iloc[position][:150], "\n")

Tabel periksa yang sama sesudah bersih ringan. Baris mojibake, HTML, dan karakter tak terlihat
harus 0; URL, mention, emoji masih ada (itu urusan bersih berat).

In [ ]:
audit_after = audit_text(light_train, light_test)
compare = audit_before[["cek", "train %"]].rename(columns={"train %": "train % sebelum"})
compare["train % sesudah"] = audit_after["train %"]
compare["test % sesudah"] = audit_after["test %"]
compare

In [ ]:
# HAPUS saat lomba: cek bahwa semua kotoran demo hilang
residue = light_train.str.contains(r"‚Ä|¬©|&amp|<br>|\u200b|SUMBER", regex=True)
print("CEK demo: baris train yang masih kotor =", int(residue.sum()), "(harus 0)")

## 4. Bersih BERAT (hanya untuk TF-IDF / model berbasis kata)
Tiap langkah membuang informasi, jadi tiap langkah diukur di bagian 5. Urutan penting: huruf kecil
→ URL/mention → emoji jadi kata → huruf berulang → simbol → slang → stopword → stemming.
Kata negasi (`tidak`, `bukan`, ...) tidak ikut dibuang stopword (`KEEP_WORDS`).

In [ ]:
KEEP_WORDS = {"tidak", "enggak", "belum", "bukan", "kurang", "jangan", "tapi", "sangat", "banget"}

slang_table = pd.concat([pd.read_csv(f"{DICTIONARY_FOLDER}/kamus_alay_salsabila.csv"),
                         pd.read_csv(f"{DICTIONARY_FOLDER}/kamus_slang.csv")])
SLANG = dict(zip(slang_table["slang"], slang_table["baku"]))   # baris belakang menimpa depan
STOPWORDS = set(StopWordRemoverFactory().get_stop_words()) - KEEP_WORDS
stemmer = StemmerFactory().create_stemmer()
ROOTS = {}                                                     # kata -> kata dasar (di-stem sekali)


def tokenize_links(text):
    text = re.sub(r"https?://\S+|www\.\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " mentiontoken ", text)
    return re.sub(r"#(\w+)", r"\1", text)                       # #kecewa -> kecewa


def emoji_to_words(text):
    return emoji.demojize(text, language="id", delimiters=(" emoji_", " "))


def squeeze_repeats(text):
    return re.sub(r"([a-z])\1{2,}", r"\1", text)                # mantaaappp -> mantap


def drop_symbols(text):
    return " ".join(re.sub(r"[^\w\s]", " ", text).split())      # angka tetap


print(drop_symbols(squeeze_repeats(emoji_to_words(tokenize_links("mantaaappp 😡 @a #kecewa!!")))))

In [ ]:
def number_token(text):
    return re.sub(r"\d+", " angkatoken ", text)


def replace_slang(text):
    words = []
    for word in text.split():
        words.append(SLANG.get(word, word))
    return " ".join(words)


def drop_stopwords(text):
    words = []
    for word in text.split():
        if word not in STOPWORDS:
            words.append(word)
    return " ".join(words)


def stem_words(text):
    words = []
    for word in text.split():
        if word not in ROOTS:
            ROOTS[word] = stemmer.stem(word)
        words.append(ROOTS[word])
    return " ".join(words)


print(stem_words(drop_stopwords(replace_slang("gak bisa dibuka tp udh bayar makanannya"))))

In [ ]:
HEAVY_STEPS = [                             # beri '#' pada langkah yang tidak dipakai
    ("huruf kecil", str.lower),
    ("url+mention", tokenize_links),
    ("emoji", emoji_to_words),
    ("huruf berulang", squeeze_repeats),
    ("simbol", drop_symbols),
    # ("angka -> token", number_token),     # bila nilai angkanya tidak penting (harga, nomor)
    ("slang", replace_slang),               # hanya teks Indonesia informal
    ("stopword", drop_stopwords),           # hanya teks Indonesia
    ("stemming", stem_words),               # hanya teks Indonesia, lambat di kosakata besar
]

start = time.time()
heavy_train, heavy_stages = apply_steps(light_train, HEAVY_STEPS, "train")
heavy_test, _ = apply_steps(light_test, HEAVY_STEPS, "test")
print("detik:", round(time.time() - start, 1), "| kata unik yang di-stem:", len(ROOTS))
print("RINGAN:", light_train.iloc[0][:150])
print("BERAT :", heavy_train.iloc[0][:150])
pd.DataFrame(step_log).pivot(index="langkah", columns="bagian", values="baris berubah")

## 5. Ukur: langkah mana yang menaikkan skor?
Model (TF-IDF kata + karakter → LinearSVC, sama dengan `baseline_oof`) dan fold sama untuk semua
tahap; yang berubah hanya teksnya. Tahap kumulatif (tiap baris = semua langkah di atasnya).

Cara membaca kolom `arti`:
- `tidak beda` (selisih ≤ simpangan antar-fold): langkah itu tidak berguna untuk skor, matikan
  kalau lambat. Ini hasil yang paling sering.
- `naik`: pertahankan. `turun`: matikan, KECUALI langkah `pola berulang` yang membuang bocoran
  label yang tidak ada di test (bagian 2): di situ skor turun justru skor yang jujur.

Target angka (regresi): ganti `LinearSVC` → `Ridge()`, `StratifiedKFold` → `KFold`, dan `SCORING`.

In [ ]:
model = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                               sublinear_tf=True)),
    LinearSVC(C=0.5, class_weight="balanced"))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
picked = train.sample(min(len(train), MEASURE_ROWS), random_state=SEED).index
stages = [("mentah", raw_train)] + light_stages + heavy_stages
results = []
previous = np.nan
for name, texts in stages:
    scores = cross_val_score(model, texts.loc[picked], train.loc[picked, TARGET], cv=folds,
                             scoring=SCORING, n_jobs=-1)
    meaning = "tidak beda"
    if name == "mentah":
        meaning = "-"
    if scores.mean() - previous > scores.std():
        meaning = "naik"
    if previous - scores.mean() > scores.std():
        meaning = "turun"
    results.append({"sampai langkah": name, "skor": round(scores.mean(), 4),
                    "simpangan": round(scores.std(), 4),
                    "selisih": round(scores.mean() - previous, 4), "arti": meaning})
    previous = scores.mean()
results = pd.DataFrame(results)
results

## 6. Kolom tabel (lewati kalau data hanya teks)
Diperiksa: semua kolom yang ada di train DAN test selain id dan teks. Kolom yang hanya ada di
train tidak disentuh dan tidak boleh jadi fitur.

- **kosong tersamar**: `-`, `?`, `n/a`, `null`, ... yang dibaca pandas sebagai teks → jadi NaN.
- **angka sebagai teks**: `Rp 1.250.000`, `750rb`, `2,5 jt`, `70 kg` → angka, di kolom yang sama.
- **tanggal sebagai teks** → tanggal + kolom baru tahun, bulan, hari dalam minggu.
- **kategori**: spasi dan huruf besar/kecil diseragamkan (` Jakarta` = `jakarta`).

Bentuk yang lebih sulit (satuan campur, `2 thn 3 bln`, typo kategori): `contoh/preprocessing/`
(`angka_satuan_tanggal`, `kategori_fuzzy`).

In [ ]:
MISSING_MARKS = ["", "-", "--", "?", "n/a", "na", "nan", "null", "none", "tidak ada", "kosong"]
SCALES = {"rb": 1e3, "ribu": 1e3, "k": 1e3, "jt": 1e6, "juta": 1e6, "miliar": 1e9}


NUMBER_SHAPE = r"(?:rp\.?|idr|[$±~])?\s*(-?\d[\d.,]*)\s*([a-z%]*)\.?"   # seluruh isi sel = angka


def parse_number(value):
    match = re.fullmatch(NUMBER_SHAPE, str(value).lower().strip())
    if match is None:
        return np.nan
    digits = match.group(1).rstrip(".,")
    if re.fullmatch(r"-?\d{1,3}(\.\d{3})+(,\d+)?", digits):      # 1.250.000,50 (Indonesia)
        digits = digits.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"-?\d{1,3}(,\d{3})+(\.\d+)?", digits):    # 1,250,000.50 (Inggris)
        digits = digits.replace(",", "")
    else:
        digits = digits.replace(",", ".")                        # 2,5 -> 2.5
    if digits.count(".") > 1:
        return np.nan
    return float(digits) * SCALES.get(match.group(2), 1.0)


print(parse_number("Rp 1.250.000"), parse_number("2,5 jt"), parse_number("750rb"),
      parse_number("70,5 kg"), parse_number("-"), parse_number("12/03/2024"))

In [ ]:
def to_dates(values):
    dates = pd.to_datetime(values, errors="coerce", dayfirst=True, format="mixed")
    shaped = values.astype(str).str.contains(r"\d[/-]\d|[a-zA-Z]{3}", regex=True)
    return dates.where(shaped)          # angka polos ("2024", "5") bukan tanggal


def audit_column(column):
    values = train[column]
    as_text = values.astype(str).str.strip()
    hidden = as_text.str.lower().isin(MISSING_MARKS) & values.notna()
    real = values[values.notna() & ~hidden]
    row = {"kolom": column, "tipe": str(values.dtype), "kosong %": share(values.isna()),
           "kosong tersamar %": share(hidden), "unik": real.nunique(), "angka %": 0.0,
           "tanggal %": 0.0, "test tak ada di train %": 0.0, "contoh": str(real.iloc[0])[:30]}
    if values.dtype == object:
        row["angka %"] = share(real.apply(parse_number).notna())
        row["tanggal %"] = share(to_dates(real).notna())
        row["test tak ada di train %"] = share(~test[column].isin(values) & test[column].notna())
    return row


OTHER_COLS = []
for column in train.columns:
    if column in test.columns and column not in [ID_COL] + TEXT_COLS:
        OTHER_COLS.append(column)
column_rows = []
for column in OTHER_COLS:
    column_rows.append(audit_column(column))
column_audit = pd.DataFrame(column_rows)
column_audit

In [ ]:
suggest_number, suggest_date, suggest_category = [], [], []
for row in column_rows:
    if row["tipe"] != "object":
        continue
    if row["tanggal %"] >= 80:
        suggest_date.append(row["kolom"])
    elif row["angka %"] >= 80:
        suggest_number.append(row["kolom"])
    elif row["unik"] <= 0.5 * len(train):
        suggest_category.append(row["kolom"])
    else:
        print("kolom teks bebas (pertimbangkan masuk TEXT_COLS):", row["kolom"])

NUMBER_COLS = suggest_number         # GANTI bila tebakan salah, mis. ["harga", "berat"]
DATE_COLS = suggest_date             # GANTI bila tebakan salah
CATEGORY_COLS = suggest_category     # GANTI bila tebakan salah
print("angka:", NUMBER_COLS, "| tanggal:", DATE_COLS, "| kategori:", CATEGORY_COLS)

In [ ]:
def clean_columns(table):
    for column in NUMBER_COLS + DATE_COLS + CATEGORY_COLS:
        as_text = table[column].astype(str).str.strip()
        table[column] = table[column].mask(as_text.str.lower().isin(MISSING_MARKS))
    for column in NUMBER_COLS:
        table[column] = table[column].apply(parse_number)
    for column in DATE_COLS:
        dates = to_dates(table[column])
        table[column] = dates
        table[f"{column}_tahun"] = dates.dt.year
        table[f"{column}_bulan"] = dates.dt.month
        table[f"{column}_hari"] = dates.dt.dayofweek
    for column in CATEGORY_COLS:
        tidy = table[column].str.lower().str.strip()
        table[column] = tidy.str.replace(r"\s+", " ", regex=True)
    return table


print(clean_columns(train.head(4).copy())[OTHER_COLS])

In [ ]:
def add_meta(table, raw):
    letters = raw.str.count(r"[A-Za-z]").clip(lower=1)
    table["meta_n_huruf"] = raw.str.len()
    table["meta_n_kata"] = raw.str.split().str.len()
    table["meta_rasio_kapital"] = raw.str.count(r"[A-Z]") / letters
    table["meta_n_seru"] = raw.str.count("!")
    table["meta_n_tanya"] = raw.str.count(r"\?")
    table["meta_n_angka"] = raw.str.count(r"\d+")
    table["meta_n_url"] = raw.str.count(r"https?://\S+")
    table["meta_n_emoji"] = raw.apply(emoji.emoji_count)
    return table


add_meta(train.head(3).copy(), raw_train.head(3)).filter(like="meta_")

## Susun tabel hasil
Tiap kolom teks dibersihkan sendiri-sendiri: kolom asli = versi ringan, `<kolom>_tfidf` = versi
berat. Fitur `meta_*` dihitung dari teks mentah (huruf besar dan tanda baca masih utuh).

In [ ]:
def clean_table(table, raw):
    table = add_meta(table.copy(), raw)
    for column in TEXT_COLS:
        texts = table[column].fillna("").astype(str)
        for _, function in LIGHT_STEPS:
            texts = texts.apply(function)
        table[column] = texts
        for _, function in HEAVY_STEPS:
            texts = texts.apply(function)
        table[f"{column}_tfidf"] = texts
    return clean_columns(table)


train_out = clean_table(train, raw_train)
test_out = clean_table(test, raw_test)
# angka kosong: model pohon (LightGBM) menerima NaN apa adanya. Model linear butuh diisi:
# for column in NUMBER_COLS:
#     median = train_out[column].median()            # nilai dari TRAIN, dipakai juga di test
#     train_out[column] = train_out[column].fillna(median)
#     test_out[column] = test_out[column].fillna(median)
train_out.head(3)

## 7. Baris: kosong, kembar, kembar beda label (hanya train)
Membersihkan sering memunculkan kembar baru (dua teks yang tadinya beda hanya di URL atau huruf
rusak). Kembar di train membuat teks yang sama ada di lipatan latih dan validasi → skor CV terlalu
tinggi. Kalau test juga banyak berisi teks train (`test ada di train`), kembar itu memang bagian
dari soal: biarkan, dan pakai label train untuk baris test yang kembar.

In [ ]:
DROP_EMPTY = True               # buang baris train yang teksnya kosong sesudah dibersihkan
DROP_MISSING_LABEL = True       # buang baris train tanpa label
DROP_FULL_DUPLICATES = True     # buang baris train yang semua kolomnya (selain id) kembar
DROP_CONFLICTS = False          # True = buang teks kembar yang labelnya berbeda-beda

text_now = join_text(train_out)
labels_per_text = train_out.groupby(text_now)[TARGET].transform("nunique")
row_report = {
    "train awal": len(train_out),
    "teks kosong": int((text_now.str.strip() == "").sum()),
    "label kosong": int(train_out[TARGET].isna().sum()),
    "teks kembar mentah": int(raw_train.duplicated().sum()),
    "teks kembar sesudah ringan": int(text_now.duplicated().sum()),
    "teks kembar sesudah berat": int(heavy_train.duplicated().sum()),
    "baris kembar beda label": int((labels_per_text > 1).sum()),
    "test ada di train": int(join_text(test_out).isin(set(text_now)).sum()),
}
keep = pd.Series(True, index=train_out.index)
if DROP_EMPTY:
    keep = keep & (text_now.str.strip() != "")
if DROP_MISSING_LABEL:
    keep = keep & train_out[TARGET].notna()
if DROP_FULL_DUPLICATES:
    keep = keep & ~train_out.drop(columns=[ID_COL]).duplicated()
if DROP_CONFLICTS:
    keep = keep & (labels_per_text <= 1)
train_out = train_out[keep]
row_report["train akhir"] = len(train_out)
pd.Series(row_report)

## 8. Simpan
Tiga cek harus `True` sebelum file dipakai: jumlah baris test tetap, urutan id test tetap, kolom
asli masih lengkap.

In [ ]:
print("CEK baris test tetap   :", len(test_out) == len(test))
print("CEK urutan id test sama:", test_out[ID_COL].tolist() == test[ID_COL].tolist())
print("CEK kolom asli lengkap :", set(train.columns) <= set(train_out.columns))
train_out.to_csv(f"{OUTPUT_FOLDER}/train_bersih.csv", index=False)
test_out.to_csv(f"{OUTPUT_FOLDER}/test_bersih.csv", index=False)

report_lines = ["LANGKAH (baris berubah)", pd.DataFrame(step_log).to_string(index=False), "",
                f"SKOR PER LANGKAH ({SCORING})", results.to_string(index=False), "",
                "BARIS", pd.Series(row_report).to_string()]
with open(f"{OUTPUT_FOLDER}/langkah.txt", "w", encoding="utf-8") as report_file:
    report_file.write("\n".join(report_lines))

tfidf_cols = []
for column in TEXT_COLS:
    tfidf_cols.append(f"{column}_tfidf")
print("\nPakai di notebook model (sel Konfigurasi):")
print(f'  TRAIN_CSV = "{OUTPUT_FOLDER}/train_bersih.csv"')
print(f'  TEST_CSV = "{OUTPUT_FOLDER}/test_bersih.csv"')
print("  TEXT_COLS =", TEXT_COLS, " -> finetune_oof / BERT (versi ringan)")
print("  TEXT_COLS =", tfidf_cols, " -> baseline_oof / TF-IDF (versi berat), bila sel ukur naik")
print("kolom:", train_out.columns.tolist())